# LiteServe Benchmark on Google Colab (T4 GPU)

Runs the LiteServe benchmark suite on a free Colab **T4 (16 GB)** GPU and produces the numbers for the README results tables.

**Before running:** `Runtime → Change runtime type → Hardware accelerator: T4 GPU`, then `Runtime → Run all`.

It will:
1. Verify the GPU
2. Clone LiteServe and install dependencies
3. Benchmark **sequential vs. continuous batching** (throughput multiplier)
4. Run a **full HTTP load test** across concurrency levels
5. Print copy-paste-ready result tables

## Configuration

Edit these if you want a different model. The default (Mistral-7B FP16) fits a T4 for the small comparison run; the HTTP sweep uses INT4 quantization for headroom across higher concurrency.

Mistral-7B is **gated** on Hugging Face — either accept its license and set `HF_TOKEN` below, or switch `MODEL` to an open model such as `Qwen/Qwen2.5-7B`.

In [ ]:
MODEL = "mistralai/Mistral-7B-v0.1"  # or e.g. "Qwen/Qwen2.5-7B" (open, no token needed)
HF_TOKEN = ""  # optional: paste a Hugging Face token for gated models

MAX_TOKENS = 128
COMPARISON_REQUESTS = 8         # for sequential-vs-batched comparison
CONCURRENCY_LEVELS = [1, 4, 8]  # for HTTP load test
QUANTIZATION = "int4-bnb"       # server model variant: null (FP16), "int8", "int4-bnb"

## 1. Verify GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clone and install

In [ ]:
import os
if not os.path.isdir('LiteServe'):
    !git clone https://github.com/vinu008/LiteServe.git
%cd LiteServe

# Colab already ships torch; install the rest.
!pip install -q transformers accelerate sentencepiece protobuf \
    fastapi 'uvicorn[standard]' sse-starlette pyyaml prometheus-client httpx bitsandbytes
!pip install -q -e . 2>/dev/null || echo 'editable install (deps already present)'

In [ ]:
# Optional: log in for gated models (Mistral, Llama, ...)
if HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)
    print('Logged in to Hugging Face.')
else:
    print('No HF_TOKEN set — make sure MODEL is an ungated model, or it will fail to download.')

## 3. Sequential vs. continuous batching

Runs the inference engine directly (no HTTP) to measure the throughput multiplier from continuous batching. Uses FP16 here — only `COMPARISON_REQUESTS` KV-caches are live at once, so it fits a T4.

In [ ]:
!python scripts/benchmark_comparison.py --model "$MODEL" \
    --num-requests {COMPARISON_REQUESTS} --max-tokens {MAX_TOKENS}

## 4. HTTP load test across concurrency

Starts the LiteServe server (quantized variant for memory headroom), waits for it to load, then runs the load test.

In [ ]:
# Generate a Colab server config
import yaml
quant = None if QUANTIZATION in (None, 'null', '') else QUANTIZATION
cfg = {
    'server': {'host': '0.0.0.0', 'port': 8000, 'workers': 1},
    'models': [{'name': 'bench', 'path': MODEL, 'quantization': quant,
                'max_model_len': 2048, 'default': True}],
    'scheduler': {'max_batch_size': 32, 'max_waiting_time_ms': 100,
                  'memory_budget_pct': 0.9, 'preemption_policy': 'fcfs',
                  'scheduling_interval_ms': 1},
    'kv_cache': {'block_size': 16, 'swap_space_gb': 1},
    'router': {'strategy': 'quality_first', 'gpu_util_high_threshold': 0.85,
               'gpu_util_low_threshold': 0.4, 'queue_depth_threshold': 20},
    'metrics': {'enabled': False, 'port': 9090},
    'logging': {'level': 'INFO', 'format': 'text'},
}
with open('configs/colab.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('Wrote configs/colab.yaml:'); print(open('configs/colab.yaml').read())

In [ ]:
import subprocess, time, httpx

# Launch the server in the background
server = subprocess.Popen(
    ['python', '-m', 'liteserve.server.app', '--config', 'configs/colab.yaml'],
    stdout=open('server.log', 'w'), stderr=subprocess.STDOUT,
)

# Wait for the model to load and the server to become healthy
print('Waiting for server (model load can take a few minutes)...')
for i in range(120):
    try:
        r = httpx.get('http://localhost:8000/v1/health', timeout=2)
        if r.status_code == 200:
            print('Server healthy:', r.json().get('status')); break
    except Exception:
        pass
    time.sleep(5)
else:
    print('Server did not come up — tail of server.log:')
    print(''.join(open('server.log').readlines()[-30:]))

In [ ]:
conc = ' '.join(str(c) for c in CONCURRENCY_LEVELS)
!python benchmarks/benchmark.py --url http://localhost:8000 \
    --concurrency {conc} --num-requests 32 --max-tokens {MAX_TOKENS} \
    --output benchmarks/results.json

## 5. Render the README tables

In [ ]:
import json
results = json.load(open('benchmarks/results.json'))
print('| Concurrency | Throughput (tok/s) | Req/s | Latency p50 / p95 / p99 (ms) | TTFT p50 (ms) |')
print('|---|---|---|---|---|')
for s in results:
    print(f"| {s['concurrency']} | {s['throughput_tokens_per_sec']} | {s['requests_per_sec']} | "
          f"{s['latency_p50_ms']} / {s['latency_p95_ms']} / {s['latency_p99_ms']} | {s['ttft_p50_ms']} |")

In [ ]:
# Stop the server when done
server.terminate()
print('Server stopped.')